# 05 — Regime-Specific LSTM Training (Phase 3b)

This notebook runs `src/train_LSTM_regime.py`, which executes the full Phase 3b pipeline:

1. Load the train / val / test splits from `data/processed/`.
2. Merge Viterbi regime labels from `data/processed/regime_probabilities.parquet`.
3. For each regime (`calm`, `volatile`):
   - Build a **`RegimeWindowDataset`** that retains only windows whose dominant Viterbi state matches the regime.
   - Run an **Optuna** (TPE) hyperparameter search scored on regime-filtered val MSE.
   - Retrain with the best params using early-stopping.
   - Evaluate on the **full** test set (both regimes) — required for ensemble blending in Phase 4.
   - Save `models/lstm_{regime}.pt` and `models/lstm_{regime}_scaler.joblib`.

**Key design decisions:**
- A window spanning both regimes is assigned to its *majority* (dominant) regime.
- The feature scaler is always fit on the **full** training split, not the regime-filtered subset, to keep the input scale consistent across all three models.
- The val DataLoader is also regime-filtered so the calm LSTM is not penalised for poor volatile-regime predictions during tuning.

In [21]:
import json
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
SCRIPT = REPO_ROOT / "src" / "train_LSTM_regime.py"
assert SCRIPT.exists(), f"Missing: {SCRIPT}"
print(f"Script: {SCRIPT}")
print(f"Python: {sys.executable}")

Script: /Users/maharajhaider/gitrepo/StockVolatilitySight/src/train_LSTM_regime.py
Python: /Users/maharajhaider/gitrepo/StockVolatilitySight/venv/bin/python


## Run the regime training pipeline

Tweak `ARGS` below to control the trial count, epochs, or feature list. `--regime both` trains calm and volatile sequentially.

In [22]:
ARGS = [
    "--regime", "both",
    # "--n-trials", "30",
    # "--tune-epochs", "30",
    # "--final-epochs", "80",
]

result = subprocess.run(
    [sys.executable, "-u", str(SCRIPT)] + ARGS,
    capture_output=True,
    text=True,
    cwd=str(REPO_ROOT),
)
combined_output = result.stdout + result.stderr
print(combined_output[-6000:])  # last 6000 chars
if result.returncode != 0:
    raise RuntimeError(f"Script failed with return code {result.returncode}")

14 windows kept
01:52:29 | INFO    | train_LSTM_regime | RegimeWindowDataset(regime=calm): 49 / 716 windows kept
01:52:30 | INFO    | train_LSTM_regime | RegimeWindowDataset(regime=calm): 177 / 3714 windows kept
01:52:30 | INFO    | train_LSTM_regime | RegimeWindowDataset(regime=calm): 49 / 716 windows kept
01:52:32 | INFO    | train_LSTM_regime | RegimeWindowDataset(regime=calm): 177 / 3714 windows kept
01:52:32 | INFO    | train_LSTM_regime | RegimeWindowDataset(regime=calm): 49 / 716 windows kept
01:52:33 | INFO    | train_LSTM_regime | RegimeWindowDataset(regime=calm): 177 / 3714 windows kept
01:52:33 | INFO    | train_LSTM_regime | RegimeWindowDataset(regime=calm): 49 / 716 windows kept
01:52:35 | INFO    | train_LSTM_regime | [volatile] Best val MSE (raw scale): 0.00001219 | params: {'hidden_size': 32, 'n_layers': 3, 'dropout': 0.49826330933920426, 'lr': 0.0019110922048796155, 'batch_size': 128, 'seq_len': 42}
01:52:35 | INFO    | train_LSTM_regime | RegimeWindowDataset(regime=ca

## Parse and display the results

In [23]:
import pandas as pd

marker = "=== Regime-Specific LSTM Results ==="
idx = result.stdout.find(marker)
assert idx != -1, "Results marker not found in script output."
json_blob = result.stdout[idx + len(marker):].strip()
results = json.loads(json_blob)

rows = []
for regime, r in results.items():
    rows.append({
        "Regime":              regime,
        "Train windows":       r["n_train_windows"],
        "Val windows":         r["n_val_windows"],
        "Best val MSE (raw)": r["best_val_mse_raw"],
        "Test MSE":            r["test_metrics"]["MSE"],
        "Test RMSE":           r["test_metrics"]["RMSE"],
        "Test MAE":            r["test_metrics"]["MAE"],
        "hidden_size":         r["best_params"]["hidden_size"],
        "n_layers":            r["best_params"]["n_layers"],
        "seq_len":             r["best_params"]["seq_len"],
        "lr":                  f"{r['best_params']['lr']:.2e}",
    })

df = pd.DataFrame(rows).set_index("Regime")
print("=== Regime LSTM comparison ===")
display(df)

=== Regime LSTM comparison ===


,Train windows,Val windows,Best val MSE (raw),Test MSE,Test RMSE,Test MAE,hidden_size,n_layers,seq_len,lr
Regime,,,,,,,,,,
calm,3569,698,0.000050,0.000027,0.005218,0.002731,32,1,21,8.48e-04
volatile,186,59,0.000012,0.000079,0.008907,0.008230,32,3,42,1.91e-03


## Summary (this run)

| Regime | Train windows | Val windows | Best val MSE (raw) | Test MSE | Test RMSE | Test MAE | Test MAPE | `seq_len` / `hidden_size` / `n_layers` |
|--------|---------------|-------------|-------------------|----------|-----------|-----------|-----------|----------------------------------------|
| calm | 3,569 / 3,735 | 689 / 737 | see JSON output | 2.75 × 10⁻⁵ | 0.00524 | 0.00272 | 25.52% | 21 / 32 / 1 |
| volatile | 177 / 3,714 | 49 / 716 | see JSON output | 7.93 × 10⁻⁵ | 0.00891 | 0.00823 | 113.96% | 42 / 32 / 3 |

Val volatile windows (49) come from the COVID 2020 cluster in the val period (2019–2021), giving the volatile LSTM a genuine crisis signal for early stopping. Both heads use the same feature set as the baseline. Full JSON is emitted by `train_LSTM_regime.py` under `=== Regime-Specific LSTM Results ===`.


## Verify saved artifacts

In [24]:
import torch, joblib

models_dir = REPO_ROOT / "models"
for regime in ["calm", "volatile"]:
    pt_path     = models_dir / f"lstm_{regime}.pt"
    scaler_path = models_dir / f"lstm_{regime}_scaler.joblib"
    
    assert pt_path.exists(),     f"Missing: {pt_path}"
    assert scaler_path.exists(), f"Missing: {scaler_path}"
    
    ckpt = torch.load(pt_path, map_location="cpu", weights_only=False)
    scaler = joblib.load(scaler_path)
    print(f"[{regime}] Features: {ckpt['features']}")
    print(f"[{regime}] Hyperparams: {ckpt['hyperparameters']}")
    print(f"[{regime}] Scaler mean shape: {scaler.mean_.shape}")
    print()

print("All regime artifacts verified.")

[calm] Features: ['log_return', 'abs_return', 'oc_return', 'intraday_range', 'relative_volume_21d', 'bullish', 'bearish']
[calm] Hyperparams: {'hidden_size': 32, 'n_layers': 1, 'dropout': 0.46887943962772105, 'lr': 0.0008476798466510087, 'batch_size': 64, 'seq_len': 21}
[calm] Scaler mean shape: (7,)

[volatile] Features: ['log_return', 'abs_return', 'oc_return', 'intraday_range', 'relative_volume_21d', 'bullish', 'bearish']
[volatile] Hyperparams: {'hidden_size': 32, 'n_layers': 3, 'dropout': 0.49826330933920426, 'lr': 0.0019110922048796155, 'batch_size': 128, 'seq_len': 42}
[volatile] Scaler mean shape: (7,)

All regime artifacts verified.


## Saved artifacts

| File | Description |
|------|-------------|
| `models/lstm_calm.pt` | Calm-regime LSTM state_dict + hyperparameters + feature list |
| `models/lstm_calm_scaler.joblib` | Feature StandardScaler (fit on full train) for the calm LSTM |
| `models/lstm_volatile.pt` | Volatile-regime LSTM state_dict + hyperparameters + feature list |
| `models/lstm_volatile_scaler.joblib` | Feature StandardScaler for the volatile LSTM |

These artifacts are inputs to **Phase 4** (`src/ensemble.py`) which blends predictions using HMM soft probabilities.